# Daily Practice — 2026-10-02 — Statistics: Is That Model Improvement Real or Noise?

**Dataset:** [Diabetes](https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset) — 442 real patient records (age, sex, BMI, blood pressure, six blood serum measurements) with a quantitative measure of diabetes progression one year after baseline, loaded via `sklearn.datasets.load_diabetes` (a real clinical dataset from a 1989 study, not a synthetic one).

## Problem statement

A teammate opens a PR swapping the shipped `Ridge` regressor for a `GradientBoostingRegressor`, claiming it's "better." The PR description shows exactly one number: the candidate's MAE is a few points lower than the baseline's MAE on the held-out test set. As the ML quality reviewer, you know a single point estimate on ~130 test rows can look like an improvement purely from which rows happened to land in the test split. Your job is to build a small statistical testing toolkit that answers two separate questions before you approve the PR:

1. **Is the gap distinguishable from sampling noise?** — statistical significance, via a paired bootstrap confidence interval and a paired permutation test on the per-sample absolute errors.
2. **Is the gap big enough to matter in practice?** — practical significance, via a minimum-improvement threshold you set yourself.

## What you should produce

- Working implementations of the five functions scaffolded below.
- A printed verdict (`APPROVE` / `HOLD` / `REJECT`) showing the bootstrap CI, the permutation p-value, and the practical threshold used to reach it.
- A histogram of the bootstrapped mean-difference distribution with the 95% CI marked on it.
- A short written answer (in the cell provided, below the plot) explaining what would happen to the CI width if the test set had only 30 rows instead of ~130, and why that matters for trusting a single train/test split comparison.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

RANDOM_STATE = 42

## Load and inspect the data

Target is a continuous "disease progression one year after baseline" score — higher means more progression.

In [ ]:
data = load_diabetes(as_frame=True)
X, y = data.data, data.target
print(X.shape)
print(X.columns.tolist())
print(y.describe())

## Your task

Fill in the TODOs below. Signatures are given — don't change them, the solution cells assume these names.

In [ ]:
# TODO: split X, y into X_train, X_test, y_train, y_test
# - 70/30 split
# - random_state=RANDOM_STATE
# (no stratify here — this is a regression target, not a class label)

X_train, X_test, y_train, y_test = None, None, None, None

In [ ]:
def fit_models(X_train, y_train):
    """TODO: fit and return (baseline, candidate).
    - baseline: sklearn Ridge(alpha=1.0, random_state=RANDOM_STATE)
    - candidate: sklearn GradientBoostingRegressor(n_estimators=200, max_depth=2,
      learning_rate=0.05, random_state=RANDOM_STATE)
    Fit both on X_train, y_train.
    """
    raise NotImplementedError

In [ ]:
def per_sample_abs_errors(model, X, y):
    """TODO: return a 1-D numpy array of |y_true - y_pred| for every row in X.
    This is the raw per-sample signal everything below is built on — the mean
    of this array is the model's MAE, but keeping it per-sample (instead of
    collapsing to one number immediately) is what lets us bootstrap and
    permute in the next two functions.
    """
    raise NotImplementedError

In [ ]:
def bootstrap_ci_diff_mean(errors_a, errors_b, n_boot=10000, ci=0.95, seed=RANDOM_STATE):
    """TODO: paired bootstrap CI for mean(errors_a - errors_b).

    errors_a and errors_b are per-sample errors for the SAME test rows (paired),
    so resample row indices once per bootstrap draw and apply that draw to both
    arrays together — never resample errors_a and errors_b independently, that
    would throw away the pairing and inflate the apparent uncertainty.

    Return (ci_low, ci_high, point_estimate, boot_means) where boot_means is the
    array of all n_boot bootstrap means (you need it for the plot later).
    """
    raise NotImplementedError

In [ ]:
def permutation_test_diff_mean(errors_a, errors_b, n_perm=10000, seed=RANDOM_STATE):
    """TODO: paired permutation test for H0: mean(errors_a) == mean(errors_b).

    For each permutation, independently flip (with 50% probability) whether
    each paired (errors_a[i], errors_b[i]) is swapped, then recompute the
    difference of means. The two-sided p-value is the fraction of permuted
    |differences| that are >= the observed |difference| (add 1 to numerator
    and denominator to avoid a p-value of exactly 0).

    Return (p_value, observed_diff).
    """
    raise NotImplementedError

In [ ]:
def decide_on_claim(ci_low, ci_high, p_value, practical_threshold, alpha=0.05):
    """TODO: combine statistical and practical significance into one verdict.

    - statistically_significant: p_value < alpha
    - ci_excludes_zero: the CI doesn't straddle 0
    - practically_meaningful: abs((ci_low + ci_high) / 2) >= practical_threshold

    Return (verdict, statistically_significant, practically_meaningful) where
    verdict is one of:
    - "APPROVE: improvement is statistically significant and practically meaningful."
    - "HOLD: statistically significant but too small to matter in practice."
    - "REJECT: not statistically distinguishable from noise at this sample size."
    (third case covers: not significant, or CI straddles 0)
    """
    raise NotImplementedError

In [ ]:
# TODO: put it together.
# 1. Call fit_models, then per_sample_abs_errors for baseline and candidate on X_test/y_test.
#    Print both MAEs (the mean of each error array).
# 2. Call bootstrap_ci_diff_mean on (errors_baseline, errors_candidate).
# 3. Call permutation_test_diff_mean on the same pair.
# 4. Call decide_on_claim with practical_threshold=3.0 and print the verdict.
# 5. Plot a histogram of boot_means with vertical lines at ci_low and ci_high.

baseline_model, candidate_model = None, None
errors_baseline, errors_candidate = None, None
ci_low, ci_high, point_estimate, boot_means = None, None, None, None
p_value, observed_diff = None, None
verdict = None

### Write-up

TODO: in 3–5 sentences, explain what would happen to the width of the bootstrap CI if the
test set had only 30 rows instead of ~130 (same models, same underlying gap). Why does that
matter when a teammate hands you a single train/test split and one MAE number as proof their
model is better?

*(your answer here)*

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE
)

def fit_models(X_train, y_train):
    baseline = Ridge(alpha=1.0, random_state=RANDOM_STATE)
    baseline.fit(X_train, y_train)
    candidate = GradientBoostingRegressor(
        n_estimators=200, max_depth=2, learning_rate=0.05, random_state=RANDOM_STATE
    )
    candidate.fit(X_train, y_train)
    return baseline, candidate

def per_sample_abs_errors(model, X, y):
    preds = model.predict(X)
    return np.abs(np.asarray(y) - preds)

def bootstrap_ci_diff_mean(errors_a, errors_b, n_boot=10000, ci=0.95, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n = len(errors_a)
    diffs = errors_a - errors_b
    point_estimate = diffs.mean()
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = diffs[idx].mean(axis=1)
    alpha = 1 - ci
    ci_low, ci_high = np.quantile(boot_means, [alpha / 2, 1 - alpha / 2])
    return ci_low, ci_high, point_estimate, boot_means

def permutation_test_diff_mean(errors_a, errors_b, n_perm=10000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n = len(errors_a)
    observed_diff = errors_a.mean() - errors_b.mean()
    swap = rng.random((n_perm, n)) < 0.5
    perm_a = np.where(swap, errors_b, errors_a)
    perm_b = np.where(swap, errors_a, errors_b)
    perm_diffs = perm_a.mean(axis=1) - perm_b.mean(axis=1)
    count = (np.abs(perm_diffs) >= abs(observed_diff)).sum()
    p_value = (count + 1) / (n_perm + 1)
    return p_value, observed_diff

def decide_on_claim(ci_low, ci_high, p_value, practical_threshold, alpha=0.05):
    statistically_significant = p_value < alpha
    ci_excludes_zero = (ci_low > 0) or (ci_high < 0)
    practically_meaningful = abs((ci_low + ci_high) / 2) >= practical_threshold
    if statistically_significant and ci_excludes_zero and practically_meaningful:
        verdict = "APPROVE: improvement is statistically significant and practically meaningful."
    elif statistically_significant and ci_excludes_zero and not practically_meaningful:
        verdict = "HOLD: statistically significant but too small to matter in practice."
    else:
        verdict = "REJECT: not statistically distinguishable from noise at this sample size."
    return verdict, statistically_significant, practically_meaningful

baseline_model, candidate_model = fit_models(X_train, y_train)
errors_baseline = per_sample_abs_errors(baseline_model, X_test, y_test)
errors_candidate = per_sample_abs_errors(candidate_model, X_test, y_test)
print("baseline MAE: ", errors_baseline.mean())
print("candidate MAE:", errors_candidate.mean())

ci_low, ci_high, point_estimate, boot_means = bootstrap_ci_diff_mean(
    errors_baseline, errors_candidate
)
p_value, observed_diff = permutation_test_diff_mean(errors_baseline, errors_candidate)
verdict, sig, practical = decide_on_claim(ci_low, ci_high, p_value, practical_threshold=3.0)

print(f"observed MAE diff (baseline - candidate): {observed_diff:.3f}")
print(f"95% bootstrap CI: [{ci_low:.3f}, {ci_high:.3f}]")
print(f"permutation p-value: {p_value:.4f}")
print(verdict)

plt.figure(figsize=(7, 4))
plt.hist(boot_means, bins=40, color="#4C72B0", edgecolor="white")
plt.axvline(ci_low, color="red", linestyle="--", label="95% CI")
plt.axvline(ci_high, color="red", linestyle="--")
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("bootstrap mean(errors_baseline - errors_candidate)")
plt.ylabel("count")
plt.legend()
plt.title("Bootstrap distribution of the MAE gap")
plt.show()
```

### What you should see

On this split, the candidate's MAE is roughly 3 points lower than the baseline's — the point estimate
looks like a real win. But the 95% bootstrap CI for the gap comes out close to `[-0.4, 7.1]`, which
straddles zero, and the permutation p-value lands around `0.08` — above the conventional `0.05` cutoff.
The verdict is `REJECT`: on a 133-row test set, this particular gap is not statistically distinguishable
from noise, even though it is numerically in the direction and rough magnitude the teammate claimed.
This is exactly the kind of PR an ML quality reviewer should hold, not because the model is necessarily
worse, but because the evidence offered doesn't support the claim as stated.

### Write-up answer

With only 30 test rows instead of ~130, the standard error of the mean difference scales roughly as
`1/sqrt(n)`, so shrinking `n` by a factor of ~4.4 widens the CI by roughly `sqrt(4.4) ≈ 2.1x`. The same
observed point estimate would sit inside an even wider band that straddles zero even more comfortably,
and the permutation p-value would almost certainly rise further from noise alone — not because the
underlying models changed, but because there's far less data to pin down where the true gap lies. A
single train/test split with a small held-out set is exactly the setup where "my number went down"
claims are least trustworthy; the fix is a bigger held-out set, repeated/cross-validated splits so the
comparison is paired across folds, or a pre-registered minimum effect size — not approving on a single
point estimate.

</details>